In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

In [2]:
import gradio as gr

In [3]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    print("Google API Key not set")

OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-
Google API Key exists and begins AIzaSyAs


In [ ]:
# Connect to OpenAI, Anthropic and Google; comment out the Claude or Google lines if you're not using them

openai = OpenAI()

anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)

In [5]:

system_message = "You are a helpful assistant"

def message_gpt(prompt):
    messages = [{"role": "system", "content": system_message}, {"role": "user", "content": prompt}]
    response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages)
    return response.choices[0].message.content

In [6]:
message_gpt("What is today's date")

"Today's date is April 27, 2024."

In [7]:
def shout(text):
    print(f"shout has been called with this input {text}")
    return text.upper()

In [8]:
shout("gay")

shout has been called with this input gay


'GAY'

In [ ]:
gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode= "never").launch()

#Interface() creates a webpage when we put some text in the textbox input it goes to the callback function shout and the result is returned to the output textbox!


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [ ]:
gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode= "never").launch(share=True)
#makes this run by hugging face and does http tunneling just like ngrok/cloudflared shit

In [15]:
gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch(inbrowser=True, auth=('Ed', 'boobs'))

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


In [ ]:
message_input = gr.Textbox(label = "Your message:", info="Enter a message to be shouted", lines= 5)
message_output = gr.Textbox(label = "Thee response")

view = gr.Interface(fn=shout,
                    inputs= message_input,
                    outputs= message_output,
                    examples= ["Gay", "nigga"],
                    flagging_mode = "never")
view.launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


shout has been called with this input nigga
shout has been called with this input Gay


In [19]:
message_input = gr.Textbox(label="ask anything to gpt", info="this request is being sent to gpt4.1-mini", lines=5)
message_output = gr.Textbox(label="this is the gpt response", lines=5)

view = gr.Interface(fn=message_gpt, 
                    inputs = [message_input],
                    outputs = [message_output],
                    examples=["what is meant by the term gaylesbian"],
                    flagging_mode="never")

view.launch(share=True)                    

* Running on local URL:  http://127.0.0.1:7866
* Running on public URL: https://d9ff8295f12328f42e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [21]:
system_prompt = "You are a helpful assistant that responds in markdown without code blocks"

message_input = gr.Textbox(label="Your message:", info="Enter a message for GPT-4.1-mini", lines=7)
message_output = gr.Markdown(label="Response:")

view = gr.Interface(fn=message_gpt,
                    inputs=[message_input],
                    outputs= [message_output],
                    examples=["what is a transformer"],
                    flagging_mode='never')
view.launch()



* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


In [24]:
system_prompt = "You are a helpful assistant"

def stream_gpt_message(prompt):
    chunks = openai.chat.completions.create(model="gpt-4.1-mini", messages=[
        {"role":"system", "content": system_prompt},
        {"role":"user","content":prompt},
    ], stream=True)
    res =''
    for chunk in chunks:
        res+= chunk.choices[0].delta.content or ''
        yield res

In [25]:
message_input = gr.Textbox(label="write your message to gpt", info="message rayira puka", lines=5)
message_output = gr.Markdown(label= "gpt pukgadu response")

view = gr.Interface(
                    fn=stream_gpt_message,
                    inputs = [message_input],
                    outputs = [message_output],
                    examples = ["why is my bestfriend gay"],
                    flagging_mode = "never"
)
view.launch()


* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.


In [26]:
def shout_words():

    yield "HI"

    yield "BRO"

    yield "BYE"

In [33]:
next(shout_words())

'HI'

In [47]:
def stream_claude(prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
      ]
    stream = anthropic.chat.completions.create(
        model='claude-sonnet-4-5-20250929',
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [48]:
message_input = gr.Textbox(label="Your message:", info="Enter a message for Claude 4.5 sonnet", lines=7)
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_claude,
    title="Claude", 
    inputs=[message_input], 
    outputs=[message_output], 
    examples=[
        "Explain the Transformer architecture to a layperson",
        "Explain the Transformer architecture to an aspiring AI engineer",
        ], 
    flagging_mode="never"
    )
view.launch()

* Running on local URL:  http://127.0.0.1:7872
* To create a public link, set `share=True` in `launch()`.


In [52]:
ollama_base_url = "http://localhost:11434/v1"
def stream_local(prompt):
    ollama = OpenAI(base_url=ollama_base_url)

    chunks = ollama.chat.completions.create(
        model="gemma2:9b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ], stream = True
    )
    res =''
    for ch in chunks:
        res+=ch.choices[0].delta.content or ''
        yield res

In [53]:
def stream_model(prompt, model):
    if model=="GPT":
        result = stream_gpt_message(prompt)
    elif model=="Claude":
        result = stream_claude(prompt)
    elif model=="Gemma":
        result = stream_local(prompt)
    else:
        raise ValueError("Unknown model")
    yield from result

In [54]:
message_input = gr.Textbox(label="Your message:", info="Enter a message for the LLM", lines=7)
model_selector = gr.Dropdown(["GPT", "Claude", "Gemma"], label="Select model", value="GPT")
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_model,
    title="LLMs", 
    inputs=[message_input, model_selector], 
    outputs=[message_output], 
    examples=[
            ["Explain the Transformer architecture to a layperson", "GPT"],
            ["Explain the Transformer architecture to an aspiring AI engineer", "Claude"]
        ], 
    flagging_mode="never"
    )
view.launch()

* Running on local URL:  http://127.0.0.1:7874
* To create a public link, set `share=True` in `launch()`.


In [56]:
from scraper import fetch_website_contents

In [55]:
system_message = """
You are an assistant that analyzes the contents of a company website landing page
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
"""

In [63]:
def stream_brochure(company_name, url, model):
    yield ""
    prompt = f"Please generate a company brochure for {company_name}. Here is their landing page:\n"
    prompt += fetch_website_contents(url)
    if model=="GPT":
        result = stream_gpt_message(prompt)
    elif model=="Claude":
        result = stream_claude(prompt)
    elif model=="Gemma":
        result = stream_local(prompt)
    else:
        raise ValueError("Unknown model")
    yield from result

In [67]:
name_input = gr.Textbox(label = "Company name:")
url_input = gr.Textbox(label = "Landing page URL including http:// or https:// ")
model_selector = gr.Dropdown(["GPT", "Claude","Gemma"], label="select model", value="GPT")

message_output = gr.Textbox(label="the response brochure")

view = gr.Interface(
    fn=stream_brochure,
    title= "Brochure Generator",
    inputs= [name_input, url_input, model_selector],
    outputs = [message_output],
    examples=[
        ["Hugging Face", "https://huggingface.co", "GPT"],
        ["Edward Donner", "https://edwarddonner.com", "Claude"]
    ],
    flagging_mode= "never"
)

In [68]:
view.launch()

* Running on local URL:  http://127.0.0.1:7877
* To create a public link, set `share=True` in `launch()`.
